# In this code, we figure out which simplex transform is required by each software package.

In [4]:
import numpy as np
from tqdm.notebook import tqdm

from modulars.utils import (
    load_config,
    # centered_simplex_transform,
    anchored_softmax_simplex_forward,
)
from modulars.distributions import gen_multinomial_data_shared, lda_posterior_shared_theta

config_file = "dirichlet_config.json"
config = load_config(config_file)

theta_like = np.array(config["theta_like"], dtype=np.float32)
alpha_prior = np.array(config["alpha_prior"], dtype=np.float32)
n_cats = config["n_cats"]
N = config["N"]
total_count = config["total_count"]
max_iters = config["max_iters"]
seed = config.get("seed", 20)
shared = config.get("shared", True)

n_vec = np.full(N, total_count, dtype=int)

if shared:
    obs_counts = gen_multinomial_data_shared(
        theta_like,
        N,
        total_count,
        SEED=seed,
    )
    true_post = lda_posterior_shared_theta(obs_counts, alpha_prior)
else:
    raise NotImplementedError("we only handle the shared-theta case here")

print("obs_counts shape:", obs_counts.shape)
print("alpha_prior:", alpha_prior)
print("n_cats:", n_cats)

def _draw_latent_samples(loc, latent_param, n_samples, seed, latent_kind="cov"):
    """
    we draw latent gaussian samples from either a covariance parameterization
    or a diagonal scale parameterization.
    """
    loc = np.asarray(loc, dtype=float).reshape(-1)
    latent_param = np.asarray(latent_param, dtype=float)
    rng = np.random.default_rng(seed)

    if latent_kind == "cov":
        if latent_param.shape != (loc.shape[0], loc.shape[0]):
            raise ValueError("we expected cov to have shape (D, D)")
        return rng.multivariate_normal(mean=loc, cov=latent_param, size=int(n_samples))

    if latent_kind == "scale":
        if latent_param.shape != loc.shape:
            raise ValueError("we expected scale to have shape (D,)")
        return rng.normal(loc=loc, scale=latent_param, size=(int(n_samples), loc.shape[0]))

    raise ValueError("we expected latent_kind to be 'cov' or 'scale'")


def _simplex_sample_moments(samples):
    """
    we compute simplex-space empirical moments from sampled theta values.
    """
    samples = np.asarray(samples, dtype=float)

    if samples.ndim != 2:
        raise ValueError("we expected simplex samples to have shape (n_samples, n_cats)")

    mean = samples.mean(axis=0)
    cov = np.cov(samples, rowvar=False)
    std = samples.std(axis=0)
    return mean, cov, std


def compare_simplex_forward_fns(
        posterior_simplex_sample_fn,
        loc,
        latent_param,
        forward_fn_a,
        forward_fn_b,
        n_cats=None,
        latent_kind="cov",
        n_samples=10_000,
        seed=0,
        name_a="centered",
        name_b="anchored"):
    """
    we compare two candidate simplex forward maps by checking which one better
    matches direct posterior simplex samples from the fitted approximation.
    """
    loc = np.asarray(loc, dtype=float).reshape(-1)

    if n_cats is None:
        n_cats = loc.shape[0] + 1

    posterior_theta = np.asarray(
        posterior_simplex_sample_fn(n_samples=int(n_samples), seed=seed),
        dtype=float,
    )
    ref_mean, ref_cov, ref_std = _simplex_sample_moments(posterior_theta)

    z = _draw_latent_samples(
        loc=loc,
        latent_param=latent_param,
        n_samples=n_samples,
        seed=seed,
        latent_kind=latent_kind,
    )

    theta_a = np.asarray(forward_fn_a(z, n_cats=n_cats), dtype=float)
    theta_b = np.asarray(forward_fn_b(z, n_cats=n_cats), dtype=float)

    mean_a, cov_a, std_a = _simplex_sample_moments(theta_a)
    mean_b, cov_b, std_b = _simplex_sample_moments(theta_b)

    mean_err_a = np.linalg.norm(mean_a - ref_mean)
    mean_err_b = np.linalg.norm(mean_b - ref_mean)

    std_err_a = np.linalg.norm(std_a - ref_std)
    std_err_b = np.linalg.norm(std_b - ref_std)

    cov_err_a = np.linalg.norm(cov_a - ref_cov, ord="fro")
    cov_err_b = np.linalg.norm(cov_b - ref_cov, ord="fro")

    score_a = mean_err_a + std_err_a + cov_err_a
    score_b = mean_err_b + std_err_b + cov_err_b

    if score_a <= score_b:
        best_name = name_a
        best_forward_fn = forward_fn_a
    else:
        best_name = name_b
        best_forward_fn = forward_fn_b

    return {
        "best_name": best_name,
        "best_forward_fn": best_forward_fn,
        "reference": {
            "mean": ref_mean,
            "std": ref_std,
            "cov": ref_cov,
        },
        name_a: {
            "mean": mean_a,
            "std": std_a,
            "cov": cov_a,
            "mean_error": float(mean_err_a),
            "std_error": float(std_err_a),
            "cov_error": float(cov_err_a),
            "total_score": float(score_a),
        },
        name_b: {
            "mean": mean_b,
            "std": std_b,
            "cov": cov_b,
            "mean_error": float(mean_err_b),
            "std_error": float(std_err_b),
            "cov_error": float(cov_err_b),
            "total_score": float(score_b),
        },
    }



obs_counts shape: (10, 3)
alpha_prior: [1. 1. 1.]
n_cats: 3


# PyMC

In [2]:
import os
import pymc as pm
import pytensor

if os.environ.get("SIMPLEVI_PYTENSOR_CXX"):
    pytensor.config.cxx = os.environ["SIMPLEVI_PYTENSOR_CXX"]
def fit_pymc_once(obs_counts, alpha_prior, n_vec, max_iters=3000, seed=0):
    with pm.Model() as model:
        theta = pm.Dirichlet("theta", a=alpha_prior)
        pm.Multinomial("y", n=n_vec, p=theta, observed=np.array(obs_counts))

        advi = pm.ADVI(random_seed=seed)
        tracker = pm.callbacks.Tracker(
            mean=advi.approx.mean.eval,
            cov=advi.approx.cov.eval,
        )

        advi.fit(
            max_iters,
            callbacks=[tracker],
            progressbar=False,
            obj_n_mc=100,
        )

        final_loc = np.asarray(tracker["mean"][-1], dtype=float)
        final_cov = np.asarray(tracker["cov"][-1], dtype=float)

        def posterior_simplex_sample_fn(n_samples, seed):
            np.random.seed(seed)
            samps = advi.approx.sample(int(n_samples)).posterior.theta
            return np.asarray(samps)[0]

    return final_loc, final_cov, posterior_simplex_sample_fn


In [4]:
pymc_loc, pymc_cov, pymc_posterior_simplex_sample_fn = fit_pymc_once(
    obs_counts=obs_counts,
    alpha_prior=alpha_prior,
    n_vec=n_vec,
    max_iters=max_iters,
    seed=seed,
)

pymc_compare = compare_simplex_forward_fns(
    posterior_simplex_sample_fn=pymc_posterior_simplex_sample_fn,
    loc=pymc_loc,
    latent_param=pymc_cov,
    forward_fn_a=centered_simplex_transform,
    forward_fn_b=anchored_softmax_simplex_forward,
    n_cats=n_cats,
    latent_kind="cov",
    n_samples=50_000,
    seed=1,
    name_a="centered",
    name_b="anchored",
)

print("PyMC best forward_fn:", pymc_compare["best_name"])
print("PyMC centered score:", pymc_compare["centered"]["total_score"])
print("PyMC anchored score:", pymc_compare["anchored"]["total_score"])
print("PyMC reference mean:", pymc_compare["reference"]["mean"])
print("PyMC centered mean:", pymc_compare["centered"]["mean"])
print("PyMC anchored mean:", pymc_compare["anchored"]["mean"])
# this results in saying that the anchored mean is correct to use with pymc

Finished [100%]: Average Loss = 54.692


PyMC best forward_fn: anchored
PyMC centered score: 0.04617727100488464
PyMC anchored score: 0.00033360596043139853
PyMC reference mean: [0.3936783  0.29200006 0.31432165]
PyMC centered mean: [0.3843641  0.28494767 0.33068824]
PyMC anchored mean: [0.39372683 0.29194735 0.31432582]


# NumPyro

In [2]:
import jax
import jax.numpy as jnp
import numpyro
import numpyro.distributions as dist

from numpyro.infer import SVI, Trace_ELBO
from numpyro.infer.autoguide import AutoDiagonalNormal
from numpyro.optim import Adam


def numpyro_model(obs_counts, n_vec, alpha_prior):
    theta = numpyro.sample(
        "theta",
        dist.Dirichlet(concentration=jnp.array(alpha_prior)),
    )

    with numpyro.plate("observations", len(obs_counts)):
        numpyro.sample(
            "y",
            dist.Multinomial(total_count=n_vec, probs=theta),
            obs=obs_counts,
        )


def fit_numpyro_once(obs_counts, alpha_prior, n_vec, max_iters=3000, seed=0):
    obs_counts_jax = jnp.array(obs_counts, dtype=jnp.float32)
    n_vec_jax = jnp.array(n_vec, dtype=jnp.int32)
    alpha_prior_jax = jnp.array(alpha_prior, dtype=jnp.float32)

    guide = AutoDiagonalNormal(numpyro_model)
    optimizer = Adam(5e-4)
    svi = SVI(numpyro_model, guide, optimizer, loss=Trace_ELBO(num_particles=100))

    rng_key = jax.random.PRNGKey(seed)
    svi_result = svi.run(
        rng_key,
        max_iters,
        obs_counts_jax,
        n_vec_jax,
        alpha_prior_jax,
        progress_bar=False,
    )

    params = svi_result.params
    final_loc = np.asarray(params["auto_loc"], dtype=float)
    final_scale = np.asarray(params["auto_scale"], dtype=float)

    def posterior_simplex_sample_fn(n_samples, seed):
        samples = guide.sample_posterior(
            jax.random.PRNGKey(seed),
            params,
            sample_shape=(int(n_samples),),
        )["theta"]
        return np.asarray(samples, dtype=float)

    return final_loc, final_scale, posterior_simplex_sample_fn



In [5]:
numpyro_loc, numpyro_scale, numpyro_posterior_simplex_sample_fn = fit_numpyro_once(
    obs_counts=obs_counts,
    alpha_prior=alpha_prior,
    n_vec=n_vec,
    max_iters=max_iters,
    seed=seed,
)

numpyro_centered = compare_simplex_forward_fns(
    posterior_simplex_sample_fn=numpyro_posterior_simplex_sample_fn,
    loc=numpyro_loc,
    latent_param=numpyro_scale,
    forward_fn_a=centered_simplex_transform,
    forward_fn_b=anchored_softmax_simplex_forward,
    n_cats=n_cats,
    latent_kind="scale",
    n_samples=100_000,
    seed=1,
    name_a="centered",
    name_b="anchored",
)

numpyro_stick = compare_simplex_forward_fns(
    posterior_simplex_sample_fn=numpyro_posterior_simplex_sample_fn,
    loc=numpyro_loc,
    latent_param=numpyro_scale,
    forward_fn_a=centered_simplex_transform,
    forward_fn_b=stickbreaking_simplex_forward,
    n_cats=n_cats,
    latent_kind="scale",
    n_samples=100_000,
    seed=1,
    name_a="centered",
    name_b="stickbreaking",
)

print("centered vs anchored:", numpyro_centered["best_name"])
print("centered score:", numpyro_centered["centered"]["total_score"])
print("anchored score:", numpyro_centered["anchored"]["total_score"])

print("centered vs stickbreaking:", numpyro_stick["best_name"])
print("centered score:", numpyro_stick["centered"]["total_score"])
print("stickbreaking score:", numpyro_stick["stickbreaking"]["total_score"])
print("reference mean:", numpyro_stick["reference"]["mean"])
print("stickbreaking mean:", numpyro_stick["stickbreaking"]["mean"])
# numpyro uses a stickbreaking simplex forward map


centered vs anchored: centered
centered score: 0.03963256067551157
anchored score: 0.08283848225714933
centered vs stickbreaking: stickbreaking
centered score: 0.03963256067551157
stickbreaking score: 0.00019965235174679584
reference mean: [0.39396579 0.29226513 0.31376908]
stickbreaking mean: [0.39401237 0.29213465 0.31385297]


# TFP

In [5]:
import tensorflow as tf
import tensorflow_probability as tfp

tfd = tfp.distributions
tfb = tfp.bijectors

tf.config.experimental.enable_tensor_float_32_execution(False)
tf.keras.backend.set_floatx("float32")

def fit_tfp_once(obs_counts, alpha_prior, n_cats, max_iters=3000, seed=0):
    alpha_prior_tf = tf.constant(alpha_prior, dtype=tf.float32)
    total_counts_per_cat = tf.constant(np.sum(obs_counts, axis=0), dtype=tf.float32)

    def target_log_prob_fn(theta):
        prior_log_prob = tfd.Dirichlet(alpha_prior_tf).log_prob(theta)
        ll = tf.reduce_sum(total_counts_per_cat * tf.math.log(theta), axis=-1)
        return prior_log_prob + ll

    q_theta = tfp.experimental.vi.build_factored_surrogate_posterior(
        event_shape=[n_cats],
        bijector=tfb.IteratedSigmoidCentered(),
        dtype=tf.float32,
    )

    def get_base_distribution():
        base_dist = q_theta.distribution
        while hasattr(base_dist, "distribution"):
            base_dist = base_dist.distribution
        return base_dist

    def trace_fn(_):
        base_dist = get_base_distribution()
        return base_dist.loc, base_dist.scale

    loc_trace, scale_trace = tfp.vi.fit_surrogate_posterior(
        target_log_prob_fn=target_log_prob_fn,
        surrogate_posterior=q_theta,
        trainable_variables=q_theta.trainable_variables,
        optimizer=tf.optimizers.Adam(),
        trace_fn=trace_fn,
        num_steps=max_iters,
        sample_size=100,
        seed=seed,
        jit_compile=True,
    )

    final_loc = np.asarray(loc_trace[-1], dtype=float)
    final_scale = np.asarray(scale_trace[-1], dtype=float)

    def posterior_simplex_sample_fn(n_samples, seed):
        samples = q_theta.sample(int(n_samples), seed=seed)
        return np.asarray(samples, dtype=float)

    return final_loc, final_scale, posterior_simplex_sample_fn


In [8]:

from modulars.utils import stickbreaking_simplex_forward


tfp_loc, tfp_scale, tfp_posterior_simplex_sample_fn = fit_tfp_once(
    obs_counts=obs_counts,
    alpha_prior=alpha_prior,
    n_cats=n_cats,
    max_iters=max_iters,
    seed=seed,
)

# tfp_compare = compare_simplex_forward_fns(
#     posterior_simplex_sample_fn=tfp_posterior_simplex_sample_fn,
#     loc=tfp_loc,
#     latent_param=tfp_scale,
#     forward_fn_a=centered_simplex_transform,
#     forward_fn_b=anchored_softmax_simplex_forward,
#     n_cats=n_cats,
#     latent_kind="scale",
#     n_samples=8_000,
#     seed=1,
#     name_a="centered",
#     name_b="anchored",
# )

tfp_stick = compare_simplex_forward_fns(
    posterior_simplex_sample_fn=tfp_posterior_simplex_sample_fn,
    loc=tfp_loc,
    latent_param=tfp_scale,
    forward_fn_a=anchored_softmax_simplex_forward,
    forward_fn_b=stickbreaking_simplex_forward,
    n_cats=n_cats,
    latent_kind="scale",
    n_samples=100_000,
    seed=1,
    name_a="anchored",
    name_b="stickbreaking",
)

# print("TFP centered vs anchored:", tfp_compare["best_name"])
print("TFP centered vs stickbreaking:", tfp_stick["best_name"])

# print("TFP centered mean:", tfp_compare["centered"]["mean"])
# print("TFP anchored mean:", tfp_compare["anchored"]["mean"])
print("TFP reference mean:", tfp_stick["reference"]["mean"])
print("TFP stickbreaking mean:", tfp_stick["stickbreaking"]["mean"])

# TFP also uses stickbreaking for the forward_fn 

from modulars.utils import (
    stickbreaking_simplex_forward,
    iterated_sigmoid_centered_forward,
)

comparison = compare_simplex_forward_fns(
    posterior_simplex_sample_fn=tfp_posterior_simplex_sample_fn,
    loc=tfp_loc,
    latent_param=tfp_scale,
    forward_fn_a=stickbreaking_simplex_forward,
    forward_fn_b=iterated_sigmoid_centered_forward,
    n_cats=n_cats,
    latent_kind="scale",
    n_samples=200_000,
    seed=1,
    name_a="stickbreaking",
    name_b="iterated_sigmoid_centered",
)

print("best forward fn:", comparison["best_name"])
print("stickbreaking score:", comparison["stickbreaking"]["total_score"])
print("iterated sigmoid centered score:", comparison["iterated_sigmoid_centered"]["total_score"])
print("reference mean:", comparison["reference"]["mean"])
print("stickbreaking mean:", comparison["stickbreaking"]["mean"])
print("iterated sigmoid centered mean:", comparison["iterated_sigmoid_centered"]["mean"])


TFP centered vs stickbreaking: stickbreaking
TFP reference mean: [0.39397494 0.29230336 0.3137217 ]
TFP stickbreaking mean: [0.39384141 0.29217183 0.31398676]
best forward fn: stickbreaking
stickbreaking score: 0.00033510660734096837
iterated sigmoid centered score: 0.0003351586326018117
reference mean: [0.39390917 0.29223899 0.31385184]
stickbreaking mean: [0.39377717 0.29224453 0.3139783 ]
iterated sigmoid centered mean: [0.39377715 0.29224453 0.31397832]
